[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/okumura-kobe/kobe-data-business/blob/main/lecture07/lecture7_notebook.ipynb)

# 後期大学院演習「データで動かす社会とビジネス」
## 第7回　演習 Notebook

**テーマ：信用リスクモデルをどう評価するか**
― 改善は、一つの数字では決まらない ―

---

### この Notebook の使い方

- **STEP 1〜5 の順番に実行**してください
- 各 STEP の結果をワークシートに記録してください
- 第6回で作った Model A / B の係数をそのまま使います（再推定しない）
- 授業中はSTEP 0～5の完成コードを順番に実行し、分析結果をWorksheetに記録・解釈します。授業後は末尾のCoding Challengeでモデル評価を自分で再現してください。WorksheetとCoding Challengeは提出不要・採点対象外です。

> ⚠️ **今回行うこと**：Validation で Model A / B を検証する（「確かめる」回）
> ⚠️ **今回行わないこと**：Expected Loss・与信枠・収益・最終的な改訂判断 → 第8〜9回

**今回の中心的な問い**
> Model B は、本当に Model A より改善したのか？

---

| | |
|---|---|
| 学籍番号 | |
| 氏名 | |
| 実施日 | |

---
## Step 0　準備（変更不要）

In [ ]:
# ライブラリの読み込み（変更不要）
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.4f}'.format)
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['figure.figsize']    = (11, 4)

print('Ready')

In [ ]:
# データの読み込みとモデル再現（変更不要）
# ⚠️ 第7回では Validation sample で係数を再推定しない
#    Development 7,000件で第6回と同じ係数を再現し、Validation 3,000件に適用する
#    これが「Developmentで作り、Validationで確かめる」という検証の原則

url = "https://raw.githubusercontent.com/okumura-kobe/kobe-data-business/main/data/credit_model.csv"
df = pd.read_csv(url)

dev = df[df['split'] == 'development'].copy().reset_index(drop=True)
val = df[df['split'] == 'validation'].copy().reset_index(drop=True)

# 第6回と完全に同じ変数定義
features_A = ['age', 'credit_limit', 'latest_payment_status',
              'latest_bill_amount', 'latest_payment_amount']
features_B = features_A + ['max_delay', 'months_with_delay',
              'avg_bill_amount', 'avg_payment_amount',
              'credit_utilization', 'payment_ratio']

# Development 7,000件で係数を再現（Validation は使用しない）
X_dev_A = sm.add_constant(dev[features_A])
X_dev_B = sm.add_constant(dev[features_B])
model_A = sm.Logit(dev['default'], X_dev_A).fit(disp=0)
model_B = sm.Logit(dev['default'], X_dev_B).fit(disp=0)

print(f'Development: {len(dev):,}件  Validation: {len(val):,}件')
print(f'Model A（{len(features_A)}変数）/ Model B（{len(features_B)}変数）係数再現完了')

---
## STEP 1　Validation に PD を適用する（10分）

**Development で推定した係数を、Validation 3,000 件に適用します。**
係数は変えません。これが検証の原則です。

In [ ]:
# 1-1　Validation に pd_A / pd_B を作成する
# Development で推定した係数を Validation に適用する（Validation では再推定しない）
X_val_A = sm.add_constant(val[features_A])
X_val_B = sm.add_constant(val[features_B])

val['pd_A'] = model_A.predict(X_val_A)
val['pd_B'] = model_B.predict(X_val_B)

print('=== Validation の基本確認 ===')
print(f'件数: {len(val):,}件')
print(f'Default件数: {val["default"].sum():,}件')
print(f'実績Default率: {val["default"].mean():.4f}')
print(f'pd_A 平均: {val["pd_A"].mean():.4f}')
print(f'pd_B 平均: {val["pd_B"].mean():.4f}')

In [ ]:
# 1-2　PD の分布を確認する
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col, label in [(axes[0], 'pd_A', 'Model A'), (axes[1], 'pd_B', 'Model B')]:
    ax.hist(val[col], bins=50, color='#1E2761', alpha=0.8, edgecolor='white')
    ax.axvline(val[col].mean(), color='#C00000', linestyle='--', lw=1.5,
               label=f'Mean PD: {val[col].mean():.4f}')
    ax.set_title(f'{label} (Validation)')
    ax.set_xlabel('PD')
    ax.set_ylabel('Count')
    ax.legend()
plt.tight_layout()
plt.show()

---
## STEP 2　AUC で判別力を比較する（10分）

**AUC（Area Under the ROC Curve）**
= DefaultとNon-defaultを「順位で」正しく分けられるか（0〜1）

AUCが見ているのは確率の水準ではなく**順位の正確さ**です。
確率の水準（Calibration）は STEP 3 で確認します。

In [ ]:
# 2-1　AUC を計算する
auc_A = roc_auc_score(val['default'], val['pd_A'])
auc_B = roc_auc_score(val['default'], val['pd_B'])

print('=== AUC（Validation）===')
print(f'Model A: {auc_A:.4f}')
print(f'Model B: {auc_B:.4f}')
print(f'改善幅（B − A）: {auc_B - auc_A:+.4f}')

In [ ]:
# 2-2　ROC 曲線（2モデルを1枚に重ねる）
fpr_A, tpr_A, _ = roc_curve(val['default'], val['pd_A'])
fpr_B, tpr_B, _ = roc_curve(val['default'], val['pd_B'])

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr_A, tpr_A, color='#1E2761', lw=2, label=f'Model A (AUC={auc_A:.4f})')
ax.plot(fpr_B, tpr_B, color='#028090', lw=2, label=f'Model B (AUC={auc_B:.4f})')
ax.plot([0,1], [0,1], 'k--', lw=1, label='Random (AUC=0.5)')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve (Validation)')
ax.legend()
plt.tight_layout()
plt.show()

---
## STEP 3　Calibration で確率の水準を比較する（10分）

AUCは「順位」を見ましたが、Calibrationは「確率の水準」を見ます。

> **PD = 0.20 と推定した顧客群が、本当に約 20% Default しているか？**

PDを10分位（Decile）に分け、各ビンの「平均予測PD」と「実績Default率」を比較します。

In [ ]:
# 3-1　Calibration（Model A）
val['pd_bin_A'] = pd.qcut(val['pd_A'], 10, duplicates='drop', labels=False)

cal_A = val.groupby('pd_bin_A').agg(
    n=('default', 'count'),
    avg_pd=('pd_A', 'mean'),
    actual_dr=('default', 'mean')
).reset_index()
cal_A['差（実績 - 予測）'] = cal_A['actual_dr'] - cal_A['avg_pd']

print('=== Calibration：Model A（Validation）===')
print(cal_A.round(4).to_string(index=False))

In [ ]:
# 3-2　Calibration（Model B）
val['pd_bin_B'] = pd.qcut(val['pd_B'], 10, duplicates='drop', labels=False)

cal_B = val.groupby('pd_bin_B').agg(
    n=('default', 'count'),
    avg_pd=('pd_B', 'mean'),
    actual_dr=('default', 'mean')
).reset_index()
cal_B['差（実績 - 予測）'] = cal_B['actual_dr'] - cal_B['avg_pd']

print('=== Calibration：Model B（Validation）===')
print(cal_B.round(4).to_string(index=False))

---
## STEP 4　Cut-off を変えて誤判別を見る（10分）

PD に Cut-off を設けて、**高リスク（PD ≥ threshold）**と**低リスク（PD < threshold）**に分類します。
Cut-off を変えると、FN と FP のバランスが変わります。

| 記号 | 意味 | 信用リスクの文脈 |
|---|---|---|
| **TP** | 実際にDefault、高リスク判定 | 適切に高リスクを捉えた |
| **TN** | 実際にNon-default、低リスク判定 | 適切に低リスクを捉えた |
| **FP** | 実際はNon-default、高リスク判定 | 実際は返済できる顧客を高リスクと誤判定 → 過度な与信抑制のリスク |
| **FN** | 実際はDefault、低リスク判定 | 実際にDefaultする顧客を低リスクと誤判定 → リスクの過小評価・過剰与信のリスク |

In [ ]:
# 4-1　Cut-off 別の誤判別（Model A / B）
thresholds = [0.20, 0.25, 0.30, 0.35, 0.40]
rows = []
for th in thresholds:
    for mn, col in [('A', 'pd_A'), ('B', 'pd_B')]:
        pred = (val[col] >= th).astype(int)
        tn, fp, fn, tp = confusion_matrix(val['default'], pred).ravel()
        rows.append({'Cut-off': th, 'Model': mn, 'TP': tp, 'TN': tn, 'FP': fp, 'FN': fn})

result = pd.DataFrame(rows)
print('=== Cut-off 別 TP / TN / FP / FN（Validation）===')
print(result.to_string(index=False))

---
## STEP 5　評価を統合する（10分）

複数の評価軸を合わせて、**「Model B は改善したと言えるか」**を整理します。

> **主たる判断を1つ選び、そう判断した理由を書いてください。**
> ① 改善の兆候あり　② 評価軸によって異なる　③ 追加検証が必要

In [ ]:
# 5-1　評価の統合サマリー
th = 0.30
rA = result[(result['Cut-off']==th) & (result['Model']=='A')].iloc[0]
rB = result[(result['Cut-off']==th) & (result['Model']=='B')].iloc[0]

print('=== 評価軸の統合（Validation）===')
print(f'評価軸              Model A    Model B    変化')
print(f'AUC                {auc_A:.4f}    {auc_B:.4f}    {auc_B-auc_A:+.4f}')
print(f'FN（th={th}）      {int(rA["FN"]):>6}     {int(rB["FN"]):>6}     {int(rB["FN"]-rA["FN"]):>+6}')
print(f'FP（th={th}）      {int(rA["FP"]):>6}     {int(rB["FP"]):>6}     {int(rB["FP"]-rA["FP"]):>+6}')
print()
print('Calibrationの比較はワークシートに記録してください。')

In [ ]:
# 5-2　評価の統合メモ（コメントに記録してください）
#
# 【主たる判断】（① 改善の兆候あり / ② 評価軸によって異なる / ③ 追加検証が必要）
#
#
# 【判断の理由】
#
#
# 【何が改善し、何がまだ判断できないか】
#
# 改善：
#
# 未判断：
#


---
## まとめ

### 今日確認したこと

| STEP | 内容 |
|---|---|
| STEP 1 | Validation に pd_A / pd_B を適用した（係数は再推定しない） |
| STEP 2 | AUC / ROC 曲線で判別力を比較した |
| STEP 3 | Calibration で確率の水準を比較した |
| STEP 4 | Cut-off を変えて FN / FP のトレードオフを確認した |
| STEP 5 | 複数の評価軸を統合し、判断の根拠を整理した |

### 今回の核心

> **モデルの「良さ」は、一つの尺度では測れない。**
> **一つの数字は、モデルの一つの側面にすぎない。**

---

### 第8回の予告：PD を損失・収益・判断に変える

- Expected Loss（EL = PD × LGD × EAD）
- PD の違いが与信枠・貸出残高にどう影響するか

**事前課題**
- 今日の統合評価表（STEP 5）をワークシートに記録しておく
- BEEF+ の第8回配布資料を読む

---
## Coding Challenge：Validation sample で Model A・B の AUC を自分で計算する

> **授業外の自習課題です。提出不要・採点対象外です。**（想定時間：30〜45分）
> 授業内の STEP 0〜5 の実行には影響しません。

授業中は完成コードを実行して、評価結果を解釈しました。ここでは、STEP 2 で行った **AUC の計算** を、自分でコードを書いて再現します。

**前提**：STEP 0〜1 を実行して、Validation sample の `val`（3,000件）に `pd_A`・`pd_B` を作成済みであること。

### 課題

`val` を使い、次の処理を自分で記述してください。

1. Validation sample の実績 Default を取得する（`y_cc`）
2. Model A・Model B の予測 PD を取得する（`pd_A_cc`・`pd_B_cc`）
3. `roc_auc_score()` で両モデルの AUC を計算する（`auc_A_cc`・`auc_B_cc`）
4. AUC の差（Model B − Model A）を計算する（`auc_diff_cc`）
5. 判別力について、結果を簡潔に解釈する（コメントで1〜2行）

> 授業内の `auc_A`・`auc_B` を上書きしないよう、変数名は上の括弧内の名前を使ってください。

### ヒント

- `roc_auc_score` は STEP 0 で読み込み済みです
- `roc_auc_score(実績のDefault, 予測PD)` の順に渡します
- 使うのは **Validation sample（`val`）** です。第6回のように Development を使うのではありません
- AUC は「順位」を見る指標です。PD を 0 / 1 に変換してから渡す必要はありません

授業内の STEP 2 のコードを見ながら、自分の言葉でコードを組み立ててみてください。

In [ ]:
# Coding Challenge：ここに自分でコードを書いてください
# （変数名：y_cc / pd_A_cc / pd_B_cc / auc_A_cc / auc_B_cc / auc_diff_cc）

# 1. Validation sample の実績 Default を取得する


# 2. Model A・Model B の予測 PD を取得する


# 3. roc_auc_score() で両モデルの AUC を計算する


# 4. AUC の差（Model B − Model A）を計算する


# 5. 判別力について、結果を簡潔に解釈する（コメントで1〜2行）
#


### 自己確認

上のセルで `y_cc`・`pd_A_cc`・`pd_B_cc`・`auc_A_cc`・`auc_B_cc`・`auc_diff_cc` を作ってから、次のセルを実行してください。次の5点を確認します。

- 実績 Default が Validation の 3,000 件で、Default 件数が 664 件である
- 予測 PD（Model A・B）がそれぞれ 3,000 件である
- Model A の AUC が約 0.717 である
- Model B の AUC が約 0.759 である
- AUC の差（B − A）が約 +0.042 である

> AUC は**順位の正しさ**を見る指標です。AUC が高いことだけで、PD の水準（Calibration）や Cut-off での誤判別が良いとは言えません。

In [ ]:
# Coding Challenge 自己確認（変更不要）
# y_cc / pd_A_cc / pd_B_cc / auc_A_cc / auc_B_cc / auc_diff_cc を作ってから実行してください
try:
    y_arr  = np.asarray(y_cc)
    pa_arr = np.asarray(pd_A_cc, dtype=float)
    pb_arr = np.asarray(pd_B_cc, dtype=float)
    a_val  = float(auc_A_cc)
    b_val  = float(auc_B_cc)
    d_val  = float(auc_diff_cc)
except NameError:
    print('必要な変数がまだありません。上のセルで作成してから実行してください。')
else:
    def mark(ok): return 'OK' if ok else '要確認'
    print(f'実績Default       : {len(y_arr):,}件・Default {int(y_arr.sum()):,}件（期待値 3,000件・664件）  → {mark(len(y_arr) == 3000 and int(y_arr.sum()) == 664)}')
    print(f'予測PDの件数      : A {len(pa_arr):,}件 / B {len(pb_arr):,}件（期待値 各3,000件）  → {mark(len(pa_arr) == 3000 and len(pb_arr) == 3000)}')
    print(f'Model A の AUC    : {a_val:.4f}（期待値 約0.717）  → {mark(abs(a_val - 0.717) < 0.001)}')
    print(f'Model B の AUC    : {b_val:.4f}（期待値 約0.759）  → {mark(abs(b_val - 0.759) < 0.001)}')
    print(f'AUC の差（B − A） : {d_val:+.4f}（期待値 約+0.042）  → {mark(abs(d_val - 0.042) < 0.001)}')
    # 参考：授業内の STEP 2 を実行済みなら、AUC を比較する
    if 'auc_A' in globals() and 'auc_B' in globals():
        gap = max(abs(a_val - auc_A), abs(b_val - auc_B))
        print(f'（参考）授業内の AUC との最大差: {gap:.6f}')


---
### 解答例（自習後に確認してください）

**まず自分でコードを書いてから**、次のセルを開いてください。セルは折りたたまれています。Colab では、セルの「…」メニューの「コードを表示」で開けます。

解答例は、授業内の `auc_A`・`auc_B` を使わず、別の変数名で AUC を最初から計算し直しています。

In [ ]:
#@title 解答例（自習後に確認する） { display-mode: "form" }
# 授業内の auc_A / auc_B は使わず、別の変数名で AUC を最初から計算し直す

# 1. Validation sample の実績 Default を取得する
y_sol = val['default']

# 2. Model A・Model B の予測 PD を取得する
pd_A_sol = val['pd_A']
pd_B_sol = val['pd_B']

# 3. roc_auc_score() で両モデルの AUC を計算する
auc_A_sol = roc_auc_score(y_sol, pd_A_sol)
auc_B_sol = roc_auc_score(y_sol, pd_B_sol)

# 4. AUC の差（Model B − Model A）を計算する
auc_diff_sol = auc_B_sol - auc_A_sol

print(f'Model A の AUC: {auc_A_sol:.4f}')
print(f'Model B の AUC: {auc_B_sol:.4f}')
print(f'AUC の差（B − A）: {auc_diff_sol:+.4f}')

# 確認：件数・Default件数
print()
print(f'Validation の件数: {len(y_sol):,}件（Default {int(y_sol.sum()):,}件）')

# 5. 判別力についての解釈（例）
# ・Validation では Model B の AUC が Model A より高く、
#   Default と Non-default を順位で分ける力（判別力）は Model B の方が高い。
# ・ただし AUC は順位だけを見る指標である。
#   確率の水準（Calibration）や Cut-off での誤判別も合わせて確認する必要がある。
